# 01 · Run all benchmark suites

Runs every suite in `algogauge.toml`, appends to `history/`, and prints the headline numbers plus a Markdown table you can paste anywhere.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "python"))
import os; os.chdir(ROOT)
print("repo root:", ROOT)

In [ ]:
SKIP_PERF = False   # set True to skip flamegraphs (faster; needed if perf is unavailable)
SUITES = None       # e.g. ["tick_to_trade"]; None = all

In [ ]:
from algogauge import manifest, runner, cli
m = manifest.load(ROOT / "algogauge.toml")
results = {}
for s in m.suites:
    if SUITES and s.name not in SUITES:
        continue
    results[s.name] = runner.run_suite(s, m.defaults, ROOT, skip_perf=SKIP_PERF)

## Summary

In [ ]:
from IPython.display import Markdown, display
for name, res in results.items():
    display(Markdown(f"### {name}  (run `{res.run_id}`, perf={'on' if res.perf_ran else 'off'})\n" + cli.summary_markdown(res.records)))

## Regression check vs previous run

In [ ]:
for name in results:
    print(f"\n== {name} ==")
    cli.main(["compare", name, "--history-dir", str(ROOT / "history"), "--threshold", str(m.defaults.regression_threshold_pct)])